# Quick Peek — IEEE-CIS Fraud Detection Dataset

Before loading 590,540 transactions into MySQL, let's understand what we're working with.

This notebook reads small samples from the CSV files (NOT the full data) so it runs quickly on any laptop. We'll explore both tables, understand the columns, see missing patterns, and look at the target variable.

**Goal:** By the end, you should be comfortable with the dataset's shape, content, and quirks.

## Step 1: Import the tools

We'll use pandas for data handling and configure it to show wide tables clearly.

In [2]:
import pandas as pd

# Show all columns in DataFrames (default truncates wide tables)
pd.set_option("display.max_columns", 100)

print("Libraries loaded.")
print(f"pandas version: {pd.__version__}")

Libraries loaded.
pandas version: 2.3.3


## Step 2: Peek at the transactions file

The file is ~650 MB — we don't want to load it all just to look around. We'll read only the first **5,000 rows** as a sample. This is plenty to understand structure.

> **Why 5,000?** Enough to see column patterns, dtypes, and example values without loading the full file into RAM.

In [10]:
# Read sample (NOT full file)
tx_sample = pd.read_csv("../data/raw/train_transaction.csv")

print(f"Sample shape: {tx_sample.shape}")
print(f"  Rows in sample: {tx_sample.shape[0]:,}")
print(f"  Columns: {tx_sample.shape[1]}")

Sample shape: (590540, 394)
  Rows in sample: 590,540
  Columns: 394


## Step 3: First 5 rows — what does this actually look like?

Let's see real data, not just numbers.

## Step 4: The 5 logical column groups

394 columns sounds scary. But they break into clean groups. Let's count them.

## Step 5: The target variable — how rare is fraud?

This is the most important question. Let's check the **isFraud** distribution in our sample.

> Note: This is just our 5,000-row sample. The full 590,540-row dataset may have slightly different ratios.

In [11]:
fraud_counts = tx_sample["isFraud"].value_counts()
fraud_pct = tx_sample["isFraud"].value_counts(normalize=True) * 100

print("Fraud distribution in sample:")
print(f"  Legitimate (isFraud=0): {fraud_counts[0]:,} rows ({fraud_pct[0]:.2f}%)")
print(f"  Fraud (isFraud=1):      {fraud_counts[1]:,} rows ({fraud_pct[1]:.2f}%)")
print(f"\nImbalance ratio: 1 fraud per {fraud_counts[0] // fraud_counts[1]} legitimate transactions")

Fraud distribution in sample:
  Legitimate (isFraud=0): 569,877 rows (96.50%)
  Fraud (isFraud=1):      20,663 rows (3.50%)

Imbalance ratio: 1 fraud per 27 legitimate transactions


## Step 6: Data types and missing values

Two critical questions about every column:
1. **What type is it?** (number, text, date)
2. **How much is missing?**

For wide tables, look at the most-missing columns first.

In [13]:
print ("column dtypes:")
print(tx_sample.dtypes.value_counts())


print("\n Top 10 columns with missing percentage:")
missing = (tx_sample.isnull().sum() / len(tx_sample) * 100).sort_values(ascending=False)
print(missing.head(10).round(2))

column dtypes:
float64    376
object      14
int64        4
Name: count, dtype: int64

 Top 10 columns with missing percentage:
dist2    93.63
D7       93.41
D13      89.51
D14      89.47
D12      89.04
D6       87.61
D9       87.31
D8       87.31
V153     86.12
V149     86.12
dtype: float64


## Step 7: The identity file — much smaller and partial

Now let's peek at the second file. Remember: **only ~24% of transactions** have identity data.

In [15]:
txn_identtities =pd.read_csv("../data/raw/train_identity.csv")
print(f'txn_identtities shape: {txn_identtities.shape}')
txn_identtities.iloc[:6, :6]

txn_identtities shape: (144233, 41)


,TransactionID,id_01,id_02,id_03,id_04,id_05
0,2987004,0.0,70787.0,NaN,NaN,NaN
1,2987008,-5.0,98945.0,NaN,NaN,0.0
2,2987010,-5.0,191631.0,0.0,0.0,0.0
3,2987011,-5.0,221832.0,NaN,NaN,0.0
4,2987016,0.0,7460.0,0.0,0.0,1.0
5,2987017,-5.0,61141.0,3.0,0.0,3.0


In [16]:
txn_identtities.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NotFound,NaN,-480.0,New,NotFound,166.0,NaN,542.0,144.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,Android 7.0,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,NaN,100.0,NotFound,49.0,-300.0,New,NotFound,166.0,NaN,621.0,500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,iOS 11.1.2,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,100.0,NotFound,52.0,NaN,Found,Found,121.0,NaN,410.0,142.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,NaN,100.0,NotFound,52.0,NaN,New,NotFound,225.0,NaN,176.0,507.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,0.0,100.0,NotFound,NaN,-300.0,Found,Found,166.0,15.0,529.0,575.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,Mac OS X 10_11_6,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


## Step 8: How the two tables connect

Both files share **`TransactionID`** as the key. We will join them in Day 2 using SQL.

For now, let's verify the relationship:

In [8]:
# How many sampled identity rows have a matching transaction in our sample?
matching = idn_sample["TransactionID"].isin(tx_sample["TransactionID"]).sum()

print(f"Transaction sample rows: {len(tx_sample):,}")
print(f"Identity sample rows:    {len(idn_sample):,}")
print(f"Identity rows that match a sampled transaction: {matching}")

print("\nNote: Our samples are independent slices, so overlap is partial.")
print("In Day 2, we'll do the full LEFT JOIN in SQL and see the real numbers.")

Transaction sample rows: 5,000
Identity sample rows:    5,000
Identity rows that match a sampled transaction: 962

Note: Our samples are independent slices, so overlap is partial.
In Day 2, we'll do the full LEFT JOIN in SQL and see the real numbers.


## What we learned

5 takeaways from this quick peek:

1. **2 tables**, joined by `TransactionID`
2. **394 + 41 = 435 columns total** (1 shared = 434 unique after join)
3. **Target column `isFraud`** is severely imbalanced (~3.5%)
4. **Many columns are mostly missing** — this is signal, not noise
5. **Most columns are masked** — V1 to V339 are Vesta's engineered features

We're ready to load the data into MySQL now. Tomorrow (Day 2), we'll write SQL queries to do real exploration.